### Learning Objectives
- Understand the three core data abstractions in Spark
- Know when to use each abstraction
- Perform operations on RDDs, DataFrames
- Understand performance implications

In [39]:
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os

# Create SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Module01_RDD_vs_DataFrame") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")

Spark Version: 4.1.2
App Name: Module01_RDD_vs_DataFrame


In [40]:
# 1.1 Creating RDD from list

data = [1,2,3,4,5,6,7,8,9,10,11]
rdd = spark.sparkContext.parallelize(data)

print(F"Type: {type(rdd)}")
print(f"Number of partition: {rdd.getNumPartitions()}")
print(f"Content: {rdd.collect()}")

Type: <class 'pyspark.core.rdd.RDD'>
Number of partition: 12
Content: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [41]:
# 1.2 Creating RDD from file (text file)

# Read raw text file
text_rdd = spark.sparkContext.textFile('data\customers.csv')
print(f"Type: {type(text_rdd)}")
print(f"First 5 lines: {text_rdd.take(5)}")

Type: <class 'pyspark.core.rdd.RDD'>
First 5 lines: ['customer_id,first_name,last_name,email,phone,city,state,signup_date,customer_segment,is_active', '1,Charles,Brown,charles.brown1@email.com,555-381-5012,New York,NY,2022-01-29,Regular,True', '2,Nancy,Brown,nancy.brown2@email.com,555-658-2424,Austin,TX,2022-01-28,VIP,True', '3,James,Williams,james.williams3@email.com,555-617-1434,Houston,TX,2020-08-26,Regular,False', '4,Linda,Anderson,linda.anderson4@email.com,555-928-1106,San Jose,CA,2020-10-11,Regular,False']


In [42]:
for row in text_rdd.take(5):
    print(row)

customer_id,first_name,last_name,email,phone,city,state,signup_date,customer_segment,is_active
1,Charles,Brown,charles.brown1@email.com,555-381-5012,New York,NY,2022-01-29,Regular,True
2,Nancy,Brown,nancy.brown2@email.com,555-658-2424,Austin,TX,2022-01-28,VIP,True
3,James,Williams,james.williams3@email.com,555-617-1434,Houston,TX,2020-08-26,Regular,False
4,Linda,Anderson,linda.anderson4@email.com,555-928-1106,San Jose,CA,2020-10-11,Regular,False


In [43]:
# 1.3 RDD Transformations (Lazy)

# Transformation: map - applies function to each element
squared_rdd = rdd.map(lambda x : x**2)
print(f"Squared rdd (Not executed yet) : {squared_rdd}")

# Transformation: Filter - filter elements
even_rdd = squared_rdd.filter(lambda x: x%2 == 0)
print(f"Even RDD (not executed yet): {even_rdd}")

# Transformation: flatMap - map + flatten
nested_rdd = spark.sparkContext.parallelize([[1,2],[3,4],[5,6]])
flat_rdd = nested_rdd.flatMap(lambda x: x)
print(f"Flat RDD: {flat_rdd.collect()}")


Squared rdd (Not executed yet) : PythonRDD[5] at RDD at PythonRDD.scala:58
Even RDD (not executed yet): PythonRDD[6] at RDD at PythonRDD.scala:58
Flat RDD: [1, 2, 3, 4, 5, 6]


In [44]:
# 1.4 RDD Actions (Trigger Execution)

# Action: collect - returns all elements
print(f"Collect: {squared_rdd.collect()}")

# Action: count - returns number of elements
print(f"Count: {rdd.count()}")

# Action: first - returns first element
print(f"First: {rdd.first()}")

# Action: take - returns first n elements
print(f"Take 3: {rdd.take(3)}")

# Action: reduce - reduces elements
print(f"Sum: {rdd.reduce(lambda a, b: a + b)}")

# Action: foreach - applies function to each element (no return)
rdd.foreach(lambda x: print(x))

Collect: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121]
Count: 11
First: 1
Take 3: [1, 2, 3]
Sum: 66


In [45]:
# 1.5 RDD Key-Value Operations (PairRDD)

customer_rdd = spark.sparkContext.parallelize(
    [
        ("Premium",100),("Regular",200),("Premium",150),
        ("New",50),("Regular",300),("VIP",500)
    ]
)

# GroupByKey - groups values by key
grouped = customer_rdd.groupByKey().mapValues(list)
print(f"grouped: {grouped.collect()}")

# ReduceByKey - reduce values by keys (more efficient)
reduced = customer_rdd.reduceByKey(lambda x,y:x+y)
print(f"reducebykey: {reduced.collect()}")

# SortByKey
sorted_rdd = customer_rdd.sortByKey()
print(f"sortByKey: {sorted_rdd.collect()}")

grouped: [('Premium', [100, 150]), ('Regular', [200, 300]), ('VIP', [500]), ('New', [50])]
reducebykey: [('Premium', 250), ('Regular', 500), ('VIP', 500), ('New', 50)]
sortByKey: [('New', 50), ('Premium', 100), ('Premium', 150), ('Regular', 200), ('Regular', 300), ('VIP', 500)]


## 2. DataFrame


In [46]:
# ============================================================
# 2.1 Creating DataFrame from CSV
# ============================================================

customers_df = spark.read.csv("data/customers.csv", header=True, inferSchema=True)

print(f"Type: {type(customers_df)}")
print(f"\nSchema:")
customers_df.printSchema()
print(f"\nFirst 5 rows:")
customers_df.show(5)


Type: <class 'pyspark.sql.classic.dataframe.DataFrame'>

Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: boolean (nullable = true)


First 5 rows:
+-----------+----------+---------+--------------------+------------+--------+-----+-----------+----------------+---------+
|customer_id|first_name|last_name|               email|       phone|    city|state|signup_date|customer_segment|is_active|
+-----------+----------+---------+--------------------+------------+--------+-----+-----------+----------------+---------+
|          1|   Charles|    Brown|charles.brown1@em...|555-381-5012|New York|   NY| 2022-01-29|         Regular|     true|
|          2|    

In [47]:
# 2.2 DataFrame Operations

# Select Columns
print("=== SELECT ===")
customers_df.select("first_name","last_name","email").show(5)

# Filter rows
customers_df.filter(col("customer_segment")== ("Premium")).show(5)

# Add new column
customers_df.withColumn("full_name",concat(col("first_name"),lit(" "),col("last_name"))).show(5)


=== SELECT ===
+----------+---------+--------------------+
|first_name|last_name|               email|
+----------+---------+--------------------+
|   Charles|    Brown|charles.brown1@em...|
|     Nancy|    Brown|nancy.brown2@emai...|
|     James| Williams|james.williams3@e...|
|     Linda| Anderson|linda.anderson4@e...|
|     David|Rodriguez|david.rodriguez5@...|
+----------+---------+--------------------+
only showing top 5 rows
+-----------+----------+---------+--------------------+------------+------------+-----+-----------+----------------+---------+
|customer_id|first_name|last_name|               email|       phone|        city|state|signup_date|customer_segment|is_active|
+-----------+----------+---------+--------------------+------------+------------+-----+-----------+----------------+---------+
|          5|     David|Rodriguez|david.rodriguez5@...|555-881-6514|     Chicago|   IL| 2020-08-08|         Premium|     true|
|          6|   Richard|    Brown|richard.brown6@em...|55

In [48]:
# 2.3 DataFrame Aggregations

from pyspark.sql.functions import *

customers_df.groupBy("customer_segment").agg(
    count('*').alias('Count'),
    count_distinct("city").alias("unique_city")
).show(5)

+----------------+-----+-----------+
|customer_segment|Count|unique_city|
+----------------+-----+-----------+
|         Regular| 1234|         15|
|         Premium| 1277|         15|
|             VIP| 1266|         15|
|             New| 1223|         15|
+----------------+-----+-----------+



In [49]:
# 2.4 DataFrame SQL Operations

customers_df.createOrReplaceTempView("customers")

result = spark.sql(
    """
      Select customer_segment, count(*) as Count, count(distinct city) as unique_city
      from customers
      group by customer_segment
      order by count desc
""")

result.show()

+----------------+-----+-----------+
|customer_segment|Count|unique_city|
+----------------+-----+-----------+
|         Premium| 1277|         15|
|             VIP| 1266|         15|
|         Regular| 1234|         15|
|             New| 1223|         15|
+----------------+-----+-----------+



In [50]:
# 3.1 Performance Test: RDD vs DataFrame

import time

# Read order data

orders_df = spark.read.csv('data/orders.csv',header=True,inferSchema=True)
print(f"Orders count: {orders_df.count()}")

# Dataframe approach
start_time = time.time()

result_df = orders_df.filter(col("status")=="Delivered").groupBy("customer_id").agg(
    count('*').alias('Order_count'),
    sum('total_amount').alias("Total_spent")
).orderBy(desc("Total_spent"))

result_df.show(5)

df_time = time.time() - start_time

print(f"Dataframe time: {df_time:4f} seconds")

start_time = time.time()

# Convert to RDD
orders_rdd = orders_df.rdd

# Filter, map, reduce
result_rdd = orders_rdd \
    .filter(lambda row: row["status"] == "Delivered") \
    .map(lambda row: (row["customer_id"], (1, row["total_amount"]))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .sortBy(lambda x: x[1][1], ascending=False)

result_rdd.take(5)
rdd_time = time.time() - start_time
print(f"RDD time: {rdd_time:.4f} seconds")

print(f"\nDataFrame is {rdd_time/df_time:.1f}x faster than RDD!")


Orders count: 50000
+-----------+-----------+------------------+
|customer_id|Order_count|       Total_spent|
+-----------+-----------+------------------+
|        194|         15|24696.420000000002|
|       1079|         14|24622.559999999998|
|       1267|         17|          22989.42|
|        565|         14|22291.570000000003|
|       4005|         16|22002.600000000002|
+-----------+-----------+------------------+
only showing top 5 rows
Dataframe time: 0.290707 seconds
RDD time: 9.1513 seconds

DataFrame is 31.5x faster than RDD!


In [51]:
# CLEANUP
spark.stop()
print("SparkSession stopped")

SparkSession stopped
